# Experiment 6: Regularization for Housing Price Prediction

**Objective**: Implement Lasso (L1) and Ridge (L2) regularization to overcome overfitting in Housing Price Prediction.

**Dataset**: California Housing Dataset

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("Libraries imported successfully.")

## 2. Load and Explore Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['Price'] = y

print(f"Dataset Shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

## 3. Data Preprocessing

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied successfully.")

In [ ]:
# Create polynomial features to induce overfitting potential
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Polynomial features: {X_train_poly.shape[1]}")

## 4. Baseline Model (No Regularization)

In [ ]:
# Train baseline Linear Regression (prone to overfitting with polynomial features)
baseline_model = LinearRegression()
baseline_model.fit(X_train_poly, y_train)

y_pred_baseline_train = baseline_model.predict(X_train_poly)
y_pred_baseline_test = baseline_model.predict(X_test_poly)

baseline_metrics = {
    'train_mse': mean_squared_error(y_train, y_pred_baseline_train),
    'test_mse': mean_squared_error(y_test, y_pred_baseline_test),
    'train_r2': r2_score(y_train, y_pred_baseline_train),
    'test_r2': r2_score(y_test, y_pred_baseline_test)
}

print("Baseline Model (No Regularization):")
print("="*50)
print(f"Training MSE: {baseline_metrics['train_mse']:.4f}")
print(f"Test MSE: {baseline_metrics['test_mse']:.4f}")
print(f"Training R²: {baseline_metrics['train_r2']:.4f}")
print(f"Test R²: {baseline_metrics['test_r2']:.4f}")
print(f"\nOverfitting indicator (Train R² - Test R²): {baseline_metrics['train_r2'] - baseline_metrics['test_r2']:.4f}")

## 5. Ridge Regression (L2 Regularization)

In [ ]:
# Ridge Regression with different alpha values
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results = []

print("Ridge Regression (L2 Regularization) Results:")
print("="*70)
print(f"{'Alpha':<10} {'Train MSE':<15} {'Test MSE':<15} {'Train R²':<12} {'Test R²':<12}")
print("-"*70)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly, y_train)
    
    y_pred_train = ridge.predict(X_train_poly)
    y_pred_test = ridge.predict(X_test_poly)
    
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    ridge_results.append({
        'alpha': alpha,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'model': ridge
    })
    
    print(f"{alpha:<10} {train_mse:<15.4f} {test_mse:<15.4f} {train_r2:<12.4f} {test_r2:<12.4f}")

In [ ]:
# Find best Ridge model
best_ridge = max(ridge_results, key=lambda x: x['test_r2'])
print(f"\nBest Ridge Alpha: {best_ridge['alpha']}")
print(f"Best Ridge Test R²: {best_ridge['test_r2']:.4f}")

## 6. Lasso Regression (L1 Regularization)

In [ ]:
# Lasso Regression with different alpha values
lasso_alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]
lasso_results = []

print("Lasso Regression (L1 Regularization) Results:")
print("="*70)
print(f"{'Alpha':<10} {'Train MSE':<15} {'Test MSE':<15} {'Train R²':<12} {'Test R²':<12}")
print("-"*70)

for alpha in lasso_alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_poly, y_train)
    
    y_pred_train = lasso.predict(X_train_poly)
    y_pred_test = lasso.predict(X_test_poly)
    
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Count non-zero coefficients
    n_features_used = np.sum(lasso.coef_ != 0)
    
    lasso_results.append({
        'alpha': alpha,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'n_features': n_features_used,
        'model': lasso
    })
    
    print(f"{alpha:<10} {train_mse:<15.4f} {test_mse:<15.4f} {train_r2:<12.4f} {test_r2:<12.4f}")

print("\nFeature Selection by Lasso:")
for result in lasso_results:
    print(f"  Alpha={result['alpha']}: {result['n_features']}/{X_train_poly.shape[1]} features used")

In [ ]:
# Find best Lasso model
best_lasso = max(lasso_results, key=lambda x: x['test_r2'])
print(f"\nBest Lasso Alpha: {best_lasso['alpha']}")
print(f"Best Lasso Test R²: {best_lasso['test_r2']:.4f}")

## 7. Comparison Visualization

In [ ]:
# Plot comparison of regularization effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge: R² vs Alpha
ridge_alphas = [r['alpha'] for r in ridge_results]
ridge_train_r2 = [r['train_r2'] for r in ridge_results]
ridge_test_r2 = [r['test_r2'] for r in ridge_results]

axes[0].semilogx(ridge_alphas, ridge_train_r2, 'b-o', linewidth=2, label='Train R²', markersize=8)
axes[0].semilogx(ridge_alphas, ridge_test_r2, 'r-s', linewidth=2, label='Test R²', markersize=8)
axes[0].axhline(y=baseline_metrics['train_r2'], color='b', linestyle='--', alpha=0.5, label='Baseline Train')
axes[0].axhline(y=baseline_metrics['test_r2'], color='r', linestyle='--', alpha=0.5, label='Baseline Test')
axes[0].set_xlabel('Alpha (Regularization Strength)')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Ridge Regression (L2)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Lasso: R² vs Alpha
lasso_alphas_plot = [r['alpha'] for r in lasso_results]
lasso_train_r2 = [r['train_r2'] for r in lasso_results]
lasso_test_r2 = [r['test_r2'] for r in lasso_results]

axes[1].semilogx(lasso_alphas_plot, lasso_train_r2, 'b-o', linewidth=2, label='Train R²', markersize=8)
axes[1].semilogx(lasso_alphas_plot, lasso_test_r2, 'r-s', linewidth=2, label='Test R²', markersize=8)
axes[1].axhline(y=baseline_metrics['train_r2'], color='b', linestyle='--', alpha=0.5, label='Baseline Train')
axes[1].axhline(y=baseline_metrics['test_r2'], color='r', linestyle='--', alpha=0.5, label='Baseline Test')
axes[1].set_xlabel('Alpha (Regularization Strength)')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Lasso Regression (L1)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/regularization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare coefficient magnitudes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Baseline coefficients
axes[0].bar(range(len(baseline_model.coef_[:20])), np.abs(baseline_model.coef_[:20]), color='gray')
axes[0].set_title('Baseline (No Regularization)', fontweight='bold')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('|Coefficient|')

# Ridge coefficients
axes[1].bar(range(len(best_ridge['model'].coef_[:20])), 
            np.abs(best_ridge['model'].coef_[:20]), color='steelblue')
axes[1].set_title(f'Ridge (α={best_ridge["alpha"]})', fontweight='bold')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('|Coefficient|')

# Lasso coefficients
axes[2].bar(range(len(best_lasso['model'].coef_[:20])), 
            np.abs(best_lasso['model'].coef_[:20]), color='coral')
axes[2].set_title(f'Lasso (α={best_lasso["alpha"]})', fontweight='bold')
axes[2].set_xlabel('Feature Index')
axes[2].set_ylabel('|Coefficient|')

plt.tight_layout()
plt.savefig('../data/coefficient_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Predictions comparison
y_pred_ridge = best_ridge['model'].predict(X_test_poly)
y_pred_lasso = best_lasso['model'].predict(X_test_poly)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (y_pred, title, color) in enumerate([
    (y_pred_baseline_test, f'Baseline (R²={baseline_metrics["test_r2"]:.4f})', 'gray'),
    (y_pred_ridge, f'Ridge (R²={best_ridge["test_r2"]:.4f})', 'steelblue'),
    (y_pred_lasso, f'Lasso (R²={best_lasso["test_r2"]:.4f})', 'coral')
]):
    axes[idx].scatter(y_test, y_pred, alpha=0.4, color=color, edgecolors='k', linewidth=0.2)
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
    axes[idx].set_xlabel('Actual Price')
    axes[idx].set_ylabel('Predicted Price')
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/regularization_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

In [ ]:
# Final comparison table
comparison_data = {
    'Model': ['Baseline (No Reg)', f'Ridge (α={best_ridge["alpha"]})', f'Lasso (α={best_lasso["alpha"]})'],
    'Train R²': [baseline_metrics['train_r2'], best_ridge['train_r2'], best_lasso['train_r2']],
    'Test R²': [baseline_metrics['test_r2'], best_ridge['test_r2'], best_lasso['test_r2']],
    'Overfitting Gap': [
        baseline_metrics['train_r2'] - baseline_metrics['test_r2'],
        best_ridge['train_r2'] - best_ridge['test_r2'],
        best_lasso['train_r2'] - best_lasso['test_r2']
    ],
    'Test MSE': [baseline_metrics['test_mse'], best_ridge['test_mse'], best_lasso['test_mse']]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*70)
print("EXPERIMENT 6 SUMMARY: Regularization for Housing Price Prediction")
print("="*70)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
print("\n" + "="*70)
print("Key Findings:")
print("="*70)

print("\n1. RIDGE REGRESSION (L2 Regularization):")
print("   - Adds penalty proportional to SQUARE of coefficients")
print("   - Shrinks coefficients towards zero but never exactly zero")
print("   - Keeps all features in the model")
print(f"   - Best improvement: {((best_ridge['test_r2'] - baseline_metrics['test_r2']) / abs(baseline_metrics['test_r2'])) * 100:.2f}% in test R²")

print("\n2. LASSO REGRESSION (L1 Regularization):")
print("   - Adds penalty proportional to ABSOLUTE value of coefficients")
print("   - Can shrink coefficients to exactly zero (feature selection)")
print(f"   - Features used by best Lasso: {best_lasso['n_features']}/{X_train_poly.shape[1]}")
print(f"   - Best improvement: {((best_lasso['test_r2'] - baseline_metrics['test_r2']) / abs(baseline_metrics['test_r2'])) * 100:.2f}% in test R²")

print("\n3. OVERFITTING REDUCTION:")
print(f"   - Baseline gap: {baseline_metrics['train_r2'] - baseline_metrics['test_r2']:.4f}")
print(f"   - Ridge gap: {best_ridge['train_r2'] - best_ridge['test_r2']:.4f}")
print(f"   - Lasso gap: {best_lasso['train_r2'] - best_lasso['test_r2']:.4f}")

print("\n4. WHEN TO USE:")
print("   - Ridge: When all features are important, multicollinearity present")
print("   - Lasso: When feature selection is desired, sparse solutions needed")
print("   - Both help prevent overfitting and improve generalization")